In [1]:
from pathlib import Path
from PIL import Image
import cv2

In [2]:
# drawings  hentai  neutral  porn  sexy
cls_imap = {
    v: k for k, v in enumerate(["drawings", "hentai", "neutral", "porn", "sexy"])
}

In [3]:
cls_imap

{'drawings': 0, 'hentai': 1, 'neutral': 2, 'porn': 3, 'sexy': 4}

In [5]:
exists = open('nsfw_train.txt').read().splitlines()

proot = Path("safety240722/seqing")

for i, f in enumerate(proot.rglob("*23k*/*")):
    if not f.is_file():
        continue
    if f.suffix not in ['.jpg', '.png', '.jpeg']:
        continue

    # im = cv2.imread(str(img))
    # if im is None:
    #     print(img)
    #     continue
    # im = Image.open(f)
    # im.verify()

    # try:
    #     if hasattr(im, 'is_animated') and im.is_animated:
    #         print(f)
    #         continue
    # except ValueError as e:
    #     print(e)
    #     print(f)
    #     continue

    exists.append(f"{f} {cls_imap['porn']}")

with open('nsfw_train_1.txt', 'w') as f:
    f.write('\n'.join(exists))

print(i)

7230


In [6]:
anns = []


proot = Path("safety240722/seqing")

for f in proot.rglob("*8k*/*"):
    if not f.is_file():
        continue
    if f.suffix not in ['.jpg', '.png', '.jpeg']:
        continue

    # im = cv2.imread(str(img))
    # if im is None:
    #     print(img)
    #     continue
    # im = Image.open(f)
    # im.verify()

    # try:
    #     if hasattr(im, 'is_animated') and im.is_animated:
    #         print(f)
    #         continue
    # except ValueError as e:
    #     print(e)
    #     print(f)
    #     continue

    anns.append(f"{f} {cls_imap['porn']}")

In [7]:

root = Path("raw_data")

for sub in root.iterdir():
    cls_ = cls_imap[sub.name]
    for img in (sub/'IMAGES').iterdir():
        # if not image
        if img.suffix not in ['.jpg', '.png', '.jpeg']:
            continue

        # im = cv2.imread(str(img))
        # if im is None:
        #     print(img)
        #     continue
        # # try:
        # im = Image.open(img)
        # im.verify()

        # if hasattr(im, 'is_animated') and im.is_animated:
        #     print(im)
        #     continue
        # # except:
        # #     print(img)
        # #     continue

        anns.append(f"{img} {cls_}")

In [8]:
import mmcv
a = mmcv.imfrombytes(open("safety240722/seqing/sq_8k/26252.jpg", 'rb').read())
print(repr(a))

None


In [5]:
## check corrupted images in parallel

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import numpy as np
import mmcv
from PIL import Image
from pathlib import Path

def check_image_mmcv(path):
    try:
        img = mmcv.imfrombytes(open(path, 'rb').read(), backend='cv2')
        if img is None:
            return False
        if img.size == 0:
            print(path, "size 0")
            return False
        return True
    except Exception as e:
        return False

def check_image_cv2(path):
    try:
        img = cv2.imread(path)
        if img is None:
            print('mmcv', path)
            return False
        if img.size == 0:
            print(path, "size 0")
            return False
        return True
    except Exception as e:
        return False

def check_image_pillow(path):
    try:
        img = Image.open(path)
        img.verify()
        np.array(img)
        return True
    except Exception as e:
        print('PIL', path)
        return False

def check_both(path):
    return check_image_mmcv(path) and check_image_pillow(path)

def check_images(paths):
    with ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(check_both, paths), total=len(paths)))
    return [path for path, is_valid in zip(paths, results) if not is_valid]

## test

# paths = [ann.split(' ')[0] for ann in anns]
paths = Path("/mnt/storage/user/wangruohui/mmpretrain/data/safety240722/seqing/nsfw_23k").iterdir()
paths = list(paths)
print(len(paths), paths[0])
corrupted = check_images(paths)
# print(len(corrupted))

/mnt/storage/user/wangruohui/mmpretrain/data/safety240722/seqing/nsfw_23k/0.jpg
7231


100%|██████████| 7231/7231 [01:50<00:00, 65.70it/s] 


In [13]:
corrupted

['safety240722/seqing/sq_8k/24063.jpg',
 'safety240722/seqing/sq_8k/24101.jpg',
 'safety240722/seqing/sq_8k/24102.jpg',
 'safety240722/seqing/sq_8k/24103.jpg',
 'safety240722/seqing/sq_8k/24226.jpg',
 'safety240722/seqing/sq_8k/24245.jpg',
 'safety240722/seqing/sq_8k/24348.jpg',
 'safety240722/seqing/sq_8k/24350.jpg',
 'safety240722/seqing/sq_8k/24353.jpg',
 'safety240722/seqing/sq_8k/24432.jpg',
 'safety240722/seqing/sq_8k/24509.jpg',
 'safety240722/seqing/sq_8k/24510.jpg',
 'safety240722/seqing/sq_8k/24511.jpg',
 'safety240722/seqing/sq_8k/24512.jpg',
 'safety240722/seqing/sq_8k/24513.jpg',
 'safety240722/seqing/sq_8k/24562.jpg',
 'safety240722/seqing/sq_8k/24632.jpg',
 'safety240722/seqing/sq_8k/24705.jpg',
 'safety240722/seqing/sq_8k/24716.jpg',
 'safety240722/seqing/sq_8k/25243.jpg',
 'safety240722/seqing/sq_8k/25272.jpg',
 'safety240722/seqing/sq_8k/25286.jpg',
 'safety240722/seqing/sq_8k/25287.jpg',
 'safety240722/seqing/sq_8k/25453.jpg',
 'safety240722/seqing/sq_8k/25455.jpg',


In [14]:
# remove corrupted
anns = [ann for ann in anns if ann.split(' ')[0] not in corrupted]


In [15]:
# train test split
import random
random.seed(0)
random.shuffle(anns)
n = len(anns)
train_n = int(n * 0.8)
train_anns = anns[:train_n]
test_anns = anns[train_n:]

with open("ann_train.txt", "w") as f:
    f.write("\n".join(train_anns))
with open("ann_test.txt", "w") as f:
    f.write("\n".join(test_anns))

In [16]:
from collections import Counter
print(Counter([x.split()[1] for x in train_anns]))
print(Counter([x.split()[1] for x in test_anns]))


Counter({'2': 18936, '0': 7689, '3': 7207, '4': 3350, '1': 2066})
Counter({'2': 4693, '0': 1957, '3': 1805, '4': 826, '1': 531})
